# 01 · Train a Model From Scratch  *(Instructor · GPU)*

**Approach 1 of 3.** Here we build a language model's "brain" **from zero** — random weights — and let
it learn *only* from our small corpus.

> **Set expectations honestly:** a real model is trained on *trillions* of words using enormous compute.
> We have a tiny corpus and one GPU, so our from-scratch model will be **small and not very good.**
> **That weakness is the lesson:** training from scratch needs huge data and compute. This is *why*,
> in practice, we almost always start from a pretrained model (Notebook 02) or use RAG (Notebook 03).

In [ ]:
import json, math, torch
from pathlib import Path
from transformers import (AutoTokenizer, GPT2Config, GPT2LMHeadModel,
                          Trainer, TrainingArguments, DataCollatorForLanguageModeling)
from datasets import Dataset

ARTIFACTS = Path("artifacts")
BASE_MODEL = "Qwen/Qwen3-1.7B"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

corpus = json.load(open(ARTIFACTS / "corpus.json"))
texts = [f"{e['title']}: {e['text']}" for e in corpus]
print(f"{len(texts)} documents to train on (this is TINY — that's the point).")

### A small, randomly-initialized model
We borrow the base model's tokenizer (so words map to numbers the same way), but the model itself is
a **fresh, small GPT-style network with random weights** — it knows nothing yet.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

config = GPT2Config(vocab_size=len(tokenizer), n_positions=256,
                    n_embd=256, n_layer=4, n_head=4)   # deliberately small
model = GPT2LMHeadModel(config).to(device)
print(f"From-scratch model: {model.num_parameters()/1e6:.1f}M random parameters")

In [ ]:
def tok(batch):
    return tokenizer(batch["text"], truncation=True, max_length=256)

ds = Dataset.from_dict({"text": texts}).map(tok, batched=True, remove_columns=["text"])
collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

args = TrainingArguments(output_dir=str(ARTIFACTS / "_scratch_train"),
                         num_train_epochs=40, per_device_train_batch_size=4,
                         learning_rate=3e-4, logging_steps=10, report_to=[],
                         save_strategy="no")
trainer = Trainer(model=model, args=args, train_dataset=ds, data_collator=collator)
trainer.train()

model.save_pretrained(ARTIFACTS / "scratch_model")
tokenizer.save_pretrained(ARTIFACTS / "scratch_model")
print("Saved from-scratch model to", ARTIFACTS / "scratch_model")

### See what it learned (don't expect much!)
It has only ever seen our tiny corpus, so it produces security-*flavored* text — but it's wobbly and
makes things up. That's expected for a model this small trained on so little.

In [ ]:
def scratch_generate(prompt, max_new_tokens=40):
    ids = tokenizer(prompt, return_tensors="pt").to(device)
    out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=True,
                         top_k=40, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0], skip_special_tokens=True)

print(scratch_generate("Phishing is"))
print("---")
print(scratch_generate("Multi-factor authentication"))

**Takeaway:** training from scratch = the model learns *everything* from your data. With little
data and compute, results are weak. In Notebook 02 we'll start from a model that already understands
language and just *adapt* it — far cheaper and far better.